In [1]:
!pip install langchain-core
!pip install langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 26.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.6.1
    Uninstalling langchain-core-1.6.1:
      Successfully uninstalled langchain-core-1.6.1


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from google.colab import userdata
import os

In [3]:
# 1. Define the prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI query optimization assistant for a search engine. "
        "Given the conversation history and the latest user query, rewrite the query "
        "so that it is fully self-contained, resolves any pronouns (like 'it' or 'that'), "
        "and includes necessary context for a vector database search. "
        "Output ONLY the rewritten query with no extra pleasantries or quotation marks."),
    # The placeholder waits for a list of messages assigned to 'chat_history'
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{latest_question}")
])

In [4]:
# 2. Simulate a dynamic list of previous messages
history = [
        HumanMessage(content="How do I set up a PostgreSQL database in Docker?"),
        AIMessage(content="You can use Docker Compose. Here is a sample docker-compose.yml file...")
    ]

In [5]:
# 3. Format the prompt with the history and the new question
formatted_prompt = prompt.format_prompt(
    chat_history=history,
    latest_question="How do I back it up?"
)

In [7]:
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_APIKEY')

llm = ChatOpenAI(
        model=os.getenv('OPENAI_CHAT_MODEL', 'gpt-4o-mini'),
        temperature=0,  # Temperature controls randomness (0 = deterministic, 2 = very creative)
        # For RAG, use temperature=0 to ensure consistent, factual responses
        # Reference: https://platform.openai.com/docs/guides/text-generation/how-should-i-set-the-temperature-parameter
        timeout=120,  # Increase timeout for slower connections
        max_retries=3,  # Retry on transient failures
    )

In [8]:
final_response = llm.invoke(formatted_prompt.to_messages())

In [9]:
print(f"Rewritten Search Query --> {final_response.content}")

Rewritten Search Query --> What are the steps to back up a PostgreSQL database that is running in a Docker container?
